# Merino Query Normalization Explorer

Explores query normalization for various providers: **ADM, Finance, FlightAware, Sports, Weather**.

**Sections:**
1. [Setup]() — Downloads setup files from GCS (keywords, Firefox vocab)
2. [Normalization Pipeline Code]() — Normalization Pipeline Implementation
3. [Provider Match Code]() — per-provider matching logic (similar to merino implementation)
4. [Build Indices]() — load JSON files, build pipeline, set up autocomplete dict
5. [Test Query]() - Test normalization pipeline

## Download and setup

In [1]:
!mkdir ./data

mkdir: ./data: File exists


In [2]:
!gcloud auth login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=zT5MAyAEJot5t1W15OLg0RgW6GMugO&access_type=offline&code_challenge=qUP9i1Cp5nWy7dfdr4I-vHKPzg_KdN7GAsdsCOYRhDQ&code_challenge_method=S256


You are now logged in as [vbaungally@mozilla.com].
Your current project is [moz-fx-mozsoc-ml-nonprod].  You can change this setting by running:
  $ gcloud config set project PROJECT_ID


In [ ]:
# download keywords and logs vocab
!gsutil -m cp \
  "gs://stage-mozsoc-articles/search_and_suggest/exploration_data/adm_keywords.json" \
  "gs://stage-mozsoc-articles/search_and_suggest/exploration_data/finance_tickers.json" \
  "gs://stage-mozsoc-articles/search_and_suggest/exploration_data/flightaware_airlines.json" \
  "gs://stage-mozsoc-articles/search_and_suggest/exploration_data/word_freq.csv" \
  ./data

In [ ]:
!pip install -r requirements.txt

In [5]:
from pathlib import Path
import json, math, re, unicodedata, functools, warnings
from collections import Counter
import pandas as pd

DATA_DIR = Path("./data")

ADM_JSON      = DATA_DIR / "adm_keywords.json"
FIN_JSON      = DATA_DIR / "finance_tickers.json"
FA_JSON       = DATA_DIR / "flightaware_airlines.json"
WORD_FREQ_CSV = DATA_DIR / "word_freq.csv"

try:
    import wordsegment as _wordsegment
    _WORDSEGMENT_AVAILABLE = True
except ImportError:
    _wordsegment = None
    _WORDSEGMENT_AVAILABLE = False

print("Required files:")
for label, path in [("adm_keywords.json", ADM_JSON), ("finance_tickers.json", FIN_JSON), ("flightaware_airlines.json", FA_JSON)]:
    print(f"  {label:<28} {'OK' if path.exists() else 'NOT FOUND — update DATA_DIR'}")
print("Optional files:")
print(f"  {'word_freq.csv':<28} {'OK (log-weighted prefix index)' if WORD_FREQ_CSV.exists() else 'missing — canonical-only fallback'}")
print(f"  {'wordsegment':<28} {'OK' if _WORDSEGMENT_AVAILABLE else 'not installed (pip install wordsegment) — word-split step disabled'}")

Required files:
  adm_keywords.json            OK
  finance_tickers.json         OK
  flightaware_airlines.json    OK
Optional files:
  word_freq.csv                OK (log-weighted prefix index)
  wordsegment                  OK


---
## 2. Normalization Pipeline Code

Converts query into a canonical form that matches Merino providers as much as possible. Each step returns if a match is found, which should help with latency

**Steps (in order):**

| Step | Name | Example |
|------|------|---------|
| 1 | Tier A — unicode + casefold + whitespace | `"Amazón "` → `"amazon"` |
| 2 | Exact canonical hit — return immediately | `"amazon"` → done |
| 3 | Join normalize — merge adjacent tokens | `"door dash"` → `"doordash"` |
| 4 | Word segment — split fused tokens | `"homedepot"` → `"home depot"` |
| 4b | Exhaustive split (≤2 tokens) | `"slickdeals"` → `"slick deals"` |
| 5 | Prefix complete — autocomplete last token | `"home dep"` → `"home depot"` |
| 6 | BM25 reorder — fix token order | `"prime amazon"` → `"amazon prime"` |

In [13]:
_PUNCT_MAP = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u02bc": "'",
    "\u201c": '"', "\u201d": '"', "\u201e": '"',
    "\u2013": "-", "\u2014": "-", "\u2012": "-", "\u2015": "-",
    "\u2026": "...",
    "\u00a0": " ", "\u200b": "", "\u200c": "", "\u200d": "", "\ufeff": "",
})
_URL_RE = re.compile(
    r"^[a-z0-9._%+\-]+@[a-z0-9.\-]+\.[a-z]{2,}$"
    r"|^(https?://|www\.)\S+$"
    r"|^[a-z0-9\-]+\.[a-z]{2,}(/\S*)?$",
    re.IGNORECASE,
)

# swaps flights to canonical form (e.g "123 ua" -> "ua 123")
_FA_COMPACT_SWAP = re.compile(r"^(\d{1,5})\s+([A-Za-z]{1,3})$")

# converts query to a canonical form so matching can be done
# from test data, this step doesn't do much but good to have
def tier_a(query: str) -> str:
    if not _URL_RE.match(query.strip()):
        query = unicodedata.normalize("NFKC", query)
        query = query.translate(_PUNCT_MAP)
    return re.sub(r"\s+", " ", query.casefold().strip())

# tries to join words and check against canonical words
# e.g "best buy" -> "bestbuy"
def _try_join_normalize(tokens, canonical):
    if len(tokens) < 2:
        return None
    hits = []
    for i in range(len(tokens) - 1):
        merged = tokens[i] + tokens[i + 1]
        candidate = " ".join(tokens[:i] + [merged] + tokens[i + 2:])
        if candidate in canonical:
            hits.append(candidate)
    return hits[0] if len(hits) == 1 else None


# uses wordsegment python module to segment combined words
# based on a Peter Norvig's work
# e.g "homedepot" -> "home depot"
@functools.lru_cache(maxsize=100_000)
def _ws_segment(tok):
    return " ".join(_wordsegment.segment(tok))

def _try_wordsegment_fallback(tokens, canonical):
    if not _WORDSEGMENT_AVAILABLE:
        return None
    result_tokens = list(tokens)
    for i, tok in enumerate(tokens):
        if len(tok) < 5 or tok in canonical:
            continue
        segmented = _ws_segment(tok)
        if segmented == tok:
            continue
        candidate = " ".join(result_tokens[:i] + segmented.split() + result_tokens[i + 1:])
        if candidate in canonical:
            return candidate
    return None


# very rarely wordsegment misses some words, this tries to 
# handle that
# e.g. "slickdeals" -> "slick deals"
def _try_split_token(token, canonical, max_splits=4):
    n = len(token)
    if n < 4:
        return None
    for num_splits in range(1, min(max_splits, n - 1)):
        min_seg = 2
        indices = list(range(min_seg, min_seg * (num_splits + 1), min_seg))
        if len(indices) < num_splits or indices[-1] > n - min_seg:
            continue
        while True:
            parts, prev = [], 0
            for idx in indices:
                parts.append(token[prev:idx])
                prev = idx
            parts.append(token[prev:])
            if all(len(p) >= min_seg for p in parts):
                if " ".join(parts) in canonical:
                    return " ".join(parts)
            pos = num_splits - 1
            while pos >= 0:
                remaining = num_splits - 1 - pos
                max_val = n - min_seg * (remaining + 1)
                if indices[pos] < max_val:
                    indices[pos] += 1
                    for k in range(pos + 1, num_splits):
                        indices[k] = indices[k - 1] + min_seg
                    break
                pos -= 1
            else:
                break
    return None

def _try_split_normalize(tokens, canonical):
    result_tokens = list(tokens)
    for i, tok in enumerate(tokens):
        if len(tok) < 5 or tok in canonical:
            continue
        split = _try_split_token(tok, canonical)
        if split is not None:
            candidate = " ".join(result_tokens[:i] + split.split() + result_tokens[i + 1:])
            if candidate in canonical:
                return candidate
    return None


# Prefix autocomplete, checks last word only against firefox logs vocab
# minimum query length should be >=4, occur in >= 100 sessions and
# 2x more times than previous match (e.g. amaz -> "amazon" 200 sessions, "amazing" 50 sessions)
# also have a max frequency so we don't autocomplete actual English words
# e.g. "temp" -> "temporary"
_AUTOCOMPLETE_MIN_PREFIX_LEN   = 4
_AUTOCOMPLETE_MIN_ABS_FREQ     = 100
_AUTOCOMPLETE_MIN_FREQ_RATIO   = 2.0
_AUTOCOMPLETE_COMMON_WORD_FREQ = 8_000_000

def build_prefix_index(vocab, min_prefix_len=_AUTOCOMPLETE_MIN_PREFIX_LEN):
    best, second = {}, {}
    for word, freq in vocab.items():
        if len(word) <= min_prefix_len:
            continue
        for end in range(min_prefix_len, len(word)):
            prefix = word[:end]
            if prefix not in best:
                best[prefix] = (word, freq)
            elif freq > best[prefix][1]:
                second[prefix] = best[prefix][1]
                best[prefix] = (word, freq)
            elif freq > second.get(prefix, 0):
                second[prefix] = freq
    return {p: (e[0], e[1], second.get(p, 0)) for p, e in best.items()}

def _apply_prefix_complete(query, prefix_index, allowlist):
    tokens = query.split()
    if not tokens:
        return query
    last = tokens[-1]
    if len(last) < _AUTOCOMPLETE_MIN_PREFIX_LEN or last in allowlist:
        return query
    entry = prefix_index.get(last)
    if entry is None:
        return query
    best_word, best_freq, second_freq = entry
    if best_word == last or best_freq < _AUTOCOMPLETE_MIN_ABS_FREQ:
        return query
    if second_freq > 0 and best_freq / second_freq < _AUTOCOMPLETE_MIN_FREQ_RATIO:
        return query
    if _WORDSEGMENT_AVAILABLE:
        # use wordsegment's frequency dictionary to figure out a known english word
        tok_freq = _wordsegment.UNIGRAMS.get(last)
        if tok_freq is not None and tok_freq >= _AUTOCOMPLETE_COMMON_WORD_FREQ:
            return query
    tokens[-1] = best_word
    return " ".join(tokens)


# applies BM25 algorithm to score against corpus
# handles ordering cases "depot home" -> "home depot"
# this step didn't have as much benefit as the previous ones and so can 
# be removed if needed to avoid complexity
class BM25Index:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.corpus       = corpus
        self.k1, self.b   = k1, b
        self._tokenized   = [doc.split() for doc in corpus]
        self._corpus_set  = set(corpus)
        n                 = len(self._tokenized)
        self._avgdl       = sum(len(d) for d in self._tokenized) / max(n, 1)
        self._doc_lens    = [len(d) for d in self._tokenized]
        df = {}
        for doc in self._tokenized:
            for term in set(doc):
                df[term] = df.get(term, 0) + 1
        self._idf = {t: math.log((n - c + 0.5) / (c + 0.5) + 1) for t, c in df.items()}
        self._inv = {}
        for i, doc in enumerate(self._tokenized):
            for term, count in Counter(doc).items():
                self._inv.setdefault(term, []).append((i, count))

    def get_top_reorder(self, query):
        if query in self._corpus_set:
            return None
        q_tokens = query.split()
        if len(q_tokens) < 2:
            return None
        q_sorted = sorted(q_tokens)
        scores = {}
        for term in set(q_tokens):
            if term not in self._inv:
                continue
            idf = self._idf.get(term, 0.0)
            for doc_idx, tf in self._inv[term]:
                dl = self._doc_lens[doc_idx]
                score = idf * (tf * (self.k1 + 1)) / (tf + self.k1 * (1 - self.b + self.b * dl / self._avgdl))
                scores[doc_idx] = scores.get(doc_idx, 0.0) + score
        if not scores:
            return None
        top_idx = max(scores, key=lambda i: scores[i])
        if scores[top_idx] == 0.0 or q_sorted != sorted(self._tokenized[top_idx]):
            return None
        return self.corpus[top_idx]

def _apply_bm25_reorder(query, adm_bm25, finance_bm25):
    m = _FA_COMPACT_SWAP.match(query)
    if m:
        return f"{m.group(2)} {m.group(1)}"
    adm = adm_bm25.get_top_reorder(query)
    if adm is not None:
        return adm
    fin = finance_bm25.get_top_reorder(query)
    return fin if fin is not None else query


# ── Pipeline ───────────────────────────────────────────────────────────────────
class NormalizePipeline:
    def __init__(self, canonical, adm_bm25=None, finance_bm25=None, canonical_prefix_index=None):
        self._canonical              = canonical
        self._adm_bm25               = adm_bm25
        self._fin_bm25               = finance_bm25
        self._canonical_prefix_index = canonical_prefix_index or {}
        self._canonical_words        = {w for phrase in canonical for w in phrase.split()}
        if _WORDSEGMENT_AVAILABLE:
            _wordsegment.load()
            for phrase in canonical:
                for tok in phrase.split():
                    if len(tok) >= 5 and tok not in _wordsegment.UNIGRAMS:
                        _ws_segment(tok)

    # returns as soon as there's a match
    def normalize(self, query):
        q = tier_a(query)
        if not q or q in self._canonical:
            return q
        tokens = q.split()
        j = _try_join_normalize(tokens, self._canonical)
        if j: return j
        ws = _try_wordsegment_fallback(tokens, self._canonical)
        if ws: return ws
        if len(tokens) <= 2:
            sp = _try_split_normalize(tokens, self._canonical)
            if sp:
                return _apply_bm25_reorder(sp, self._adm_bm25, self._fin_bm25) if self._adm_bm25 else sp
        if self._canonical_prefix_index and len(tokens) >= 2:
            c = _apply_prefix_complete(q, self._canonical_prefix_index, self._canonical_words)
            if c != q:
                if c in self._canonical: return c
                q = c
        if self._adm_bm25:
            q = _apply_bm25_reorder(q, self._adm_bm25, self._fin_bm25)
        return q


print("Pipeline code loaded.")

Pipeline code loaded.


## Pipeline Matching
Tries to simulate matches with existing Merino Implementation

In [14]:
# ── ADM ────────────────────────────────────────────────────────────────────────
# Baseline: lstrip + lowercase → prefix index lookup
# _adm_index is populated in Section 4 when adm_keywords.json is loaded.
_adm_index: dict = {}

def adm_match(query: str) -> str | None:
    return _adm_index.get(query.lstrip().lower())


# ── Finance ────────────────────────────────────────────────────────────────────
# Replicates merino/providers/suggest/finance/backends/polygon/utils.py
# Data dicts are populated in Section 4 from finance_tickers.json.
_stock_tickers:   dict = {}
_stock_blocklist: set  = set()
_etf_tickers:     dict = {}
_etf_blocklist:   set  = set()
_kw_to_stock:     dict = {}
_kw_to_etf:       dict = {}
_STOCK_QUERY_RE = re.compile(
    r"^(?:(?P<kw1>\w+)\s+stocks?)$|^(?:stocks?\s+(?P<kw2>\w+))$", re.IGNORECASE
)

def finance_match(query: str) -> str | None:
    q = query.strip().replace("$", "STOCK ").replace("  ", " ")  # finance normalize
    qu = q.upper()
    if qu in _stock_blocklist or qu in _etf_blocklist:
        return None
    if qu in _stock_tickers: return qu
    if qu in _etf_tickers:   return qu
    if t := _kw_to_stock.get(q):  return t
    if t := _kw_to_etf.get(q):    return ",".join(t)
    m = _STOCK_QUERY_RE.match(qu)
    if m:
        kw = m.group("kw1") or m.group("kw2")
        if kw in _stock_tickers: return kw
        if kw in _etf_tickers:   return kw
    return None


# ── FlightAware ────────────────────────────────────────────────────────────────
# Replicates merino/providers/suggest/flightaware/backends/utils.py
# _airline_name_to_code is populated in Section 4 from flightaware_airlines.json.
_airline_name_to_code: dict = {}
_FA_PAT1 = re.compile(r"\b[A-Za-z]{1,3}\s*\d{1,5}\b", re.IGNORECASE)
_FA_PAT2 = re.compile(r"\b\d\s*[A-Za-z]\s*\d{1,4}\b", re.IGNORECASE)
_FA_KW   = re.compile(r"^[A-Za-z]+(?:\s+[a-z]+){0,4}\s+\d{1,5}$", re.IGNORECASE)

def flightaware_match(query: str) -> str | None:
    q = " ".join(query.lower().split())  # fa normalize
    m = _FA_PAT1.search(q) or _FA_PAT2.search(q)
    if m:
        matched    = str(m.group(0)).strip()
        before     = q[: m.start()].strip()
        after      = q[m.end():].strip()
        remaining  = " ".join(f"{before} {after}".split())
        if not remaining or "flight status".startswith(remaining):
            return matched.replace(" ", "").upper()
    if _FA_KW.match(q):
        parts = q.split()
        code  = _airline_name_to_code.get(" ".join(parts[:-1]))
        if code:
            return code + parts[-1]
    return None


# ── Sports ─────────────────────────────────────────────────────────────────────
# Intent words inlined from merino/providers/suggest/sports/__init__.py
_SPORTS_INTENTS = {"vs", "game", "match", "fixtures", "schedule", "play",
                   "upcoming", "score", "result", "final", "live", "today", "v"}

def sports_match(query: str) -> str | None:
    words = query.split()
    if not any(w.lower() in _SPORTS_INTENTS for w in words):
        return None
    remaining = " ".join(w for w in words if w.lower() not in _SPORTS_INTENTS)
    return remaining if remaining else None


# ── Weather ────────────────────────────────────────────────────────────────────
# Word-boundary matching to prevent "hotel"->hot, "train"->rain, etc.
_WEATHER_SINGLE = frozenset([
    "weather", "forecast", "temperature", "temp", "rain", "raining", "snow", "snowing",
    "sunny", "cloudy", "overcast", "humid", "humidity", "wind", "windy", "storm",
    "hurricane", "tornado", "cold", "hot", "climate", "degrees", "celsius", "fahrenheit",
    "precipitation", "thunderstorm", "fog", "foggy", "hail", "sleet", "blizzard",
    "heatwave", "frost", "freezing", "icy",
])
_WEATHER_MULTI = frozenset(["heat wave", "ice storm", "uv index", "air quality", "pollen count"])

def weather_match(query: str) -> str | None:
    q = query.lower()
    if set(q.split()) & _WEATHER_SINGLE or any(kw in q for kw in _WEATHER_MULTI):
        return query
    return None


# ── Multi-provider helpers ─────────────────────────────────────────────────────
_SCORES = {"sports": 0.50, "adm": 0.31, "weather": 0.30, "finance": 0.29, "flightaware": 0.29}

def provider_matches(query: str) -> dict:
    return {
        "adm":         adm_match(query),
        "finance":     finance_match(query),
        "flightaware": flightaware_match(query),
        "sports":      sports_match(query),
        "weather":     weather_match(query),
    }

def winning_provider(query: str) -> str | None:
    hits = {p: _SCORES[p] for p, v in provider_matches(query).items() if v is not None}
    return max(hits, key=lambda p: hits[p]) if hits else None


print("Provider code loaded.")

Provider code loaded.


## Load Keywords / Data and Build Indices

In [15]:
# ── ADM ────────────────────────────────────────────────────────────────────────
with ADM_JSON.open() as f:
    adm_data = json.load(f)
us_desktop = next(e for e in adm_data if e["country-code"] == "US" and e["form-factor"] == "desktop")
print(f"ADM US-desktop keywords:   { len(us_desktop['search-terms']):,}")

for kw, info in us_desktop["search-terms"].items():
    _adm_index.setdefault(kw, kw)
    for partial in info["partials"]:
        _adm_index.setdefault(partial, kw)
print(f"ADM baseline prefix index: {len(_adm_index):,} entries")

# ── Finance ────────────────────────────────────────────────────────────────────
_fin = json.loads(FIN_JSON.read_text())
_stock_tickers.update(_fin["stock_tickers"])
_stock_blocklist.update(_fin["stock_blocklist"])
_etf_tickers.update(_fin["etf_tickers"])
_etf_blocklist.update(_fin["etf_blocklist"])
_kw_to_stock.update(_fin["keyword_to_stock_ticker"])
_kw_to_etf.update(_fin["keyword_to_etf_tickers"])
print(f"Finance:                   {len(_stock_tickers):,} stocks, {len(_etf_tickers):,} ETFs, {len(_kw_to_stock)+len(_kw_to_etf):,} keywords")

# ── FlightAware ────────────────────────────────────────────────────────────────
_airline_name_to_code.update(json.loads(FA_JSON.read_text())["name_to_code"])
print(f"FlightAware airlines:      {len(_airline_name_to_code):,} entries")

ADM US-desktop keywords:   4,289
ADM baseline prefix index: 27,981 entries
Finance:                   1,000 stocks, 256 ETFs, 221 keywords
FlightAware airlines:      1,242 entries


In [16]:
# This will happen on init in merino, and can be updated on a daily / weekly basis
print("Building pipeline indexes...")

canonical    = set(us_desktop["search-terms"].keys())
fin_keywords = list({*_kw_to_stock.keys(), *_kw_to_etf.keys()})
canonical.update(fin_keywords)
print(f"  canonical set:       {len(canonical):,} terms")

adm_bm25     = BM25Index(list(us_desktop["search-terms"].keys()))
finance_bm25 = BM25Index(fin_keywords)
print(f"  ADM BM25:            {len(adm_bm25.corpus):,} keywords")
print(f"  Finance BM25:        {len(finance_bm25.corpus):,} keywords")

# word_freq.csv = word frequencies derived from Firefox query logs (BigQuery)
if WORD_FREQ_CSV.exists():
    wf       = pd.read_csv(WORD_FREQ_CSV, dtype={"word": str, "freq": int})
    word_freq = dict(zip(wf["word"], wf["freq"]))
    canonical_prefix_index = build_prefix_index(word_freq)
    print(f"  prefix index:        {len(word_freq):,} words  (log-weighted from Firefox query logs)")
else:
    word_freq = {w: 1 for phrase in canonical for w in phrase.split() if len(w) > _AUTOCOMPLETE_MIN_PREFIX_LEN}
    canonical_prefix_index = build_prefix_index(word_freq)
    print(f"  prefix index:        {len(canonical_prefix_index):,} entries  (canonical-only fallback)")

pipeline = NormalizePipeline(
    canonical=canonical,
    adm_bm25=adm_bm25,
    finance_bm25=finance_bm25,
    canonical_prefix_index=canonical_prefix_index,
)
print("\nReady.")

Building pipeline indexes...
  canonical set:       4,510 terms
  ADM BM25:            4,289 keywords
  Finance BM25:        221 keywords
  prefix index:        3,176 words  (log-weighted from Firefox query logs)

Ready.


In [20]:
list(canonical_prefix_index.items())[:10]

[('goog', ('google', 63754, 0)),
 ('googl', ('google', 63754, 0)),
 ('amaz', ('amazon', 37718, 5)),
 ('amazo', ('amazon', 37718, 0)),
 ('wher', ('where', 30504, 0)),
 ('redd', ('reddit', 28262, 0)),
 ('reddi', ('reddit', 28262, 0)),
 ('coun', ('county', 23233, 1646)),
 ('count', ('county', 23233, 1646)),
 ('stat', ('state', 19373, 2483))]

## Trace Example Query

In [21]:
def trace(query: str) -> None:
    """Step-by-step pipeline trace with per-provider match results."""
    print(f"Input:  {repr(query)}")
    print()

    q, step_fired = tier_a(query), None
    if q in pipeline._canonical:
        step_fired = "exact"
    else:
        tokens = q.split()
        j = _try_join_normalize(tokens, pipeline._canonical)
        if j:
            step_fired, q = "join", j
        else:
            ws = _try_wordsegment_fallback(tokens, pipeline._canonical)
            if ws:
                step_fired, q = "wordseg", ws
            elif len(tokens) <= 2:
                sp = _try_split_normalize(tokens, pipeline._canonical)
                if sp:
                    step_fired, q = "split", sp
        if step_fired is None:
            if pipeline._canonical_prefix_index and len(tokens) >= 2:
                c = _apply_prefix_complete(q, pipeline._canonical_prefix_index, pipeline._canonical_words)
                if c != q:
                    step_fired, q = "prefix", c
            r = _apply_bm25_reorder(q, pipeline._adm_bm25, pipeline._fin_bm25)
            if r != q:
                step_fired, q = ("bm25" if step_fired is None else f"{step_fired}+bm25"), r

    normalized = pipeline.normalize(query)
    print(f"Step:   [{step_fired}]  {repr(tier_a(query))} -> {repr(normalized)}" if step_fired
          else f"Step:   (no change)  {repr(normalized)}")
    print()

    b, p = provider_matches(query), provider_matches(normalized)
    print(f"{'Provider':<14} {'Baseline':<35} {'Pipeline':<35} Change")
    print("-" * 92)
    for prov in ["adm", "finance", "flightaware", "sports", "weather"]:
        bv, pv = b.get(prov), p.get(prov)
        b_str  = (bv[:33] + ".." if bv and len(bv) > 35 else bv) or "—"
        p_str  = (pv[:33] + ".." if pv and len(pv) > 35 else pv) or "—"
        change = ("+NEW" if pv and not bv else "-LOST" if bv and not pv
                  else "~CHANGED" if bv and pv and bv != pv else "same" if bv and pv else "")
        print(f"{prov:<14} {b_str:<35} {p_str:<35} {change}")

    bw, pw = winning_provider(query), winning_provider(normalized)
    print()
    print(f"Winner: {bw or 'none'}  {'-> ' + pw if bw != pw else '(unchanged)'}")

In [24]:
query = "prime amazo"

trace(query)

Input:  'prime amazo'

Step:   [prefix+bm25]  'prime amazo' -> 'amazon prime'

Provider       Baseline                            Pipeline                            Change
--------------------------------------------------------------------------------------------
adm            —                                   amazon prime                        +NEW
finance        —                                   —                                   
flightaware    —                                   —                                   
sports         —                                   —                                   
weather        —                                   —                                   

Winner: none  -> adm


In [25]:
# ── Batch compare ──────────────────────────────────────────────────────────────
test_queries = [
    # ADM
    ("amazon",              "exact match"),
    ("prime amazon",        "BM25 reorder"),
    ("door dash",           "join -> doordash"),
    ("slickdeals",          "split -> slick deals"),
    ("home dep",            "prefix -> home depot"),
    # Finance
    ("amazon stock",        "keyword match"),
    ("dow jones",           "ETF keyword"),
    ("tsla",                "direct ticker"),
    # FlightAware
    ("ua 123",              "compact code"),
    ("united airlines 456", "airline name"),
    # Sports
    ("lakers game",         "intent word + entity"),
    ("nba score",           "league + intent"),
    # Weather
    ("boston weather",      "weather keyword"),
    ("is it raining",       "weather keyword"),
]

rows = []
for q, note in test_queries:
    norm = pipeline.normalize(q)
    bm, pm = provider_matches(q), provider_matches(norm)
    changes = [
        f"+{prov}" if pm.get(prov) and not bm.get(prov)
        else f"-{prov}" if bm.get(prov) and not pm.get(prov)
        else None
        for prov in ["adm", "finance", "flightaware", "sports", "weather"]
    ]
    rows.append({
        "query":            q,
        "note":             note,
        "normalized":       norm if norm != tier_a(q) else "(same)",
        "baseline_winner":  winning_provider(q) or "—",
        "pipeline_winner":  winning_provider(norm) or "—",
        "provider_changes": ", ".join(c for c in changes if c) or "same",
    })

pd.set_option("display.max_colwidth", 35)
pd.DataFrame(rows)

,query,note,normalized,baseline_winner,pipeline_winner,provider_changes
0,amazon,exact match,(same),adm,adm,same
1,prime amazon,BM25 reorder,amazon prime,—,adm,+adm
2,door dash,join -> doordash,(same),—,—,same
3,slickdeals,split -> slick deals,slick deals,—,adm,+adm
4,home dep,prefix -> home depot,(same),adm,adm,same
5,amazon stock,keyword match,(same),finance,finance,same
6,dow jones,ETF keyword,(same),finance,finance,same
7,tsla,direct ticker,(same),finance,finance,same
8,ua 123,compact code,(same),flightaware,flightaware,same
9,united airlines 456,airline name,(same),flightaware,flightaware,same
